# 04 Alignment Analysis

## Description
Measures discourse alignment by embedding all documents with `sentence-transformers/all-MiniLM-L6-v2`
and computing cosine similarity to the mean Russia and Ukraine document vectors.
This operationalizes the Echo/Adapt/Resist framework: documents with high similarity to Russia's
mean vector are "echoing" Russia's narrative; documents with high similarity to Ukraine are "adapting".

## Input
- `corpus/analysis_corpus_labeled.csv` – Labeled analysis corpus

## Output
- `outputs/alignment_scores_labeled.csv` – Per-document similarity scores
- `outputs/alignment_by_month_labeled.csv` – Monthly mean alignment by actor × country
- `outputs/fig_sim_to_russia_timeseries.png` – Alignment to Russia time series
- `outputs/fig_sim_to_ukraine_timeseries.png` – Alignment to Ukraine time series


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

INPUT   = "./corpus/analysis_corpus_labeled.csv"
OUT_DIR = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TARGETS = [("State","Russia"),("State","Ukraine"),("State","Georgia"),("Institution","Institution")]
COLORS  = {"Russia":"firebrick","Ukraine":"royalblue","Georgia":"seagreen","Institution":"darkorange"}


In [ ]:
def load_text(p):
    try:
        with open(p, "r", encoding="utf-8") as f:
            t = f.read()
        return (t[:6000] + "\n...\n" + t[-2000:]) if len(t) > 9000 else t
    except Exception: return ""

def mean_vec(mask, emb):
    m = emb[mask].mean(axis=0, keepdims=True)
    norm = np.linalg.norm(m, axis=1, keepdims=True)
    return m / np.where(norm == 0, 1, norm)

df = pd.read_csv(INPUT, dtype=str)
df = df[df["clean_path"].notna()].copy().reset_index(drop=True)
df["month"] = df.get("month", df.get("date","")).fillna("").astype(str).str.slice(0,7)
docs = [load_text(p) for p in df["clean_path"]]
print(f"Encoding {len(docs)} docs ...")


In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb   = model.encode(docs, batch_size=64, convert_to_numpy=True,
                     normalize_embeddings=True, show_progress_bar=True)

r_mask = df["country"].eq("Russia").values
u_mask = df["country"].eq("Ukraine").values
r_mean = mean_vec(r_mask, emb)
u_mean = mean_vec(u_mask, emb)

df["sim_to_russia"]  = (emb @ r_mean.T).reshape(-1)
df["sim_to_ukraine"] = (emb @ u_mean.T).reshape(-1)

df.to_csv(f"{OUT_DIR}/alignment_scores_labeled.csv", index=False)
print(f"Saved alignment_scores_labeled.csv")

g = df.groupby(["actor","country","month"]).agg(
    sim_to_russia=("sim_to_russia","mean"),
    sim_to_ukraine=("sim_to_ukraine","mean"),
).reset_index()
g.to_csv(f"{OUT_DIR}/alignment_by_month_labeled.csv", index=False)
print(f"Saved alignment_by_month_labeled.csv")


In [ ]:
# Visualization
for metric in ["sim_to_russia","sim_to_ukraine"]:
    plt.figure(figsize=(13,5))
    for actor, country in TARGETS:
        sub = g[(g["actor"]==actor) & (g["country"]==country)].sort_values("month")
        if sub.empty: continue
        ls = "--" if actor == "Institution" else "-"
        plt.plot(sub["month"], sub[metric],
                 label=f"{actor}-{country}",
                 color=COLORS.get(country,"gray"), linestyle=ls, linewidth=1.4)
    plt.xticks(rotation=60, ha="right", fontsize=8)
    plt.title(f"Alignment over Time: {metric}")
    plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/fig_{metric}_timeseries.png", dpi=150); plt.close()
    print(f"Saved fig_{metric}_timeseries.png")
